# 🛡️ GaudOn: Robust URL Classifier Training (V2 - Balanced)

**Problem:** The previous model overfitted on metadata like 'URL Length' and 'HTTPS'.

**Fix:** This version uses a 1:1 balanced dataset (Malicious vs Safe) and improved normalization.

In [ ]:
!pip install scikit-learn joblib python-whois tldextract python-Levenshtein httpx

In [ ]:
import httpx
import pandas as pd
import zipfile
import io
import os
import re
import random

print("📥 Gathering Balanced Dataset...")

malicious_urls = []
try:
    openphish_url = "https://openphish.com/feed.txt"
    with httpx.Client(follow_redirects=True) as client:
        malicious_urls.extend(client.get(openphish_url).text.strip().split('\n'))
    
    urlhaus_url = "https://urlhaus.abuse.ch/downloads/csv_recent/"
    with httpx.Client(follow_redirects=True) as client:
        res = client.get(urlhaus_url)
        lines = [line for line in res.text.splitlines() if not line.startswith('#')]
        urlhaus_df = pd.read_csv(io.StringIO('\n'.join(lines)), names=["id", "dateadded", "url", "url_status", "last_online", "threat", "tags", "urlhaus_link", "reporter"])
        malicious_urls.extend(urlhaus_df['url'].tolist())
except:
    pass

malicious_urls = list(set(malicious_urls))
print(f"✅ Found {len(malicious_urls)} malicious items.")

target_count = len(malicious_urls)
tranco_url = "https://tranco-list.eu/top-1m.csv.zip"
benign_urls = []
with httpx.Client(follow_redirects=True) as client:
    res = client.get(tranco_url)
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        with z.open('top-1m.csv') as f:
            tranco_df = pd.read_csv(f, names=["rank", "domain"])
            subset = tranco_df['domain'].iloc[:target_count].tolist()
            for i, d in enumerate(subset):
                scheme = "https" if i % 2 == 0 else "http"
                benign_urls.append(f"{scheme}://{d}")

print(f"✅ Balanced! {len(benign_urls)} benign items selected.")

In [ ]:
import math
import tldextract
from urllib.parse import urlparse

SHORTENERS = {"bit.ly", "goo.gl", "tinyurl.com", "t.co", "rebrand.ly", "is.gd", "buff.ly", "ow.ly"}

def calculate_entropy(string):
    if not string: return 0.0
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    return -sum([p * math.log(p) / math.log(2.0) for p in prob])

def extract_features(url):
    try:
        url = url.strip().rstrip('/')
        parsed = urlparse(url)
        ext = tldextract.extract(url)
        domain_part = ext.domain
        
        domain_age = 0
        keywords = ["verify", "secure", "otp", "login", "update", "bvn", "bank", "alert"]
        keyword_count = sum(1 for kw in keywords if kw in url.lower())
        entropy = calculate_entropy(domain_part)
        subdomain_depth = len(ext.subdomain.split('.')) if ext.subdomain else 0
        is_https = 1 if parsed.scheme == "https" else 0
        url_length = len(url)
        hyphen_count = domain_part.count('-')
        
        risky_tlds = [".xyz", ".tk", ".top", ".ml", ".ga"]
        tld_risk_score = 0.8 if any(ext.suffix.endswith(t[1:]) for t in risky_tlds) else 0.1
        
        special_char_count = sum(url.count(c) for c in ['@', '=', '?', '_', '&'])
        lookalikes = ['0', '1', '3', '5']
        num_sub = sum(domain_part.count(n) for n in lookalikes) / len(domain_part) if domain_part else 0
        
        percent_enc = url.count('%')
        path_part = url[url.find("//")+2:]
        double_slash = 1 if "//" in path_part else 0
        is_ip = 1 if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", domain_part) else 0
        
        vowels = "aeiou"
        v = sum(1 for c in domain_part.lower() if c in vowels)
        c = sum(1 for c in domain_part.lower() if c.isalpha() and c not in vowels)
        vc_ratio = v / c if c > 0 else 0.0
        
        consec = 0
        if len(domain_part) > 1:
            cur = 1
            for i in range(1, len(domain_part)):
                if domain_part[i] == domain_part[i-1]: cur += 1
                else: consec = max(consec, cur); cur = 1
            consec = max(consec, cur)
            
        is_short = 1 if f"{ext.domain}.{ext.suffix}" in ["bit.ly", "t.co", "tinyurl.com"] else 0
        non_std_port = 1 if parsed.port and parsed.port not in [80, 443] else 0
        
        return [domain_age, keyword_count, entropy, subdomain_depth, is_https, 
                url_length, hyphen_count, tld_risk_score, special_char_count, num_sub,
                percent_enc, double_slash, is_ip, vc_ratio, consec, is_short, non_std_port]
    except:
        return None

print("⚙️ Extracting...")
data, labels = [], []
for url in malicious_urls: 
    f = extract_features(url)
    if f: data.append(f); labels.append(1)
for url in benign_urls: 
    f = extract_features(url)
    if f: data.append(f); labels.append(0)
print(f"✅ Ready: {len(data)} samples.")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np

X, y = np.array(data), np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42)
clf.fit(X_train, y_train)
print(f"Accuracy: {clf.score(X_test, y_test):.4f}")

test_url = "https://www.google.com"
test_feat = extract_features(test_url)
prob = clf.predict_proba([test_feat])[0]
print(f"\nTest (google.com): Phishing Prob: {prob[1]:.4f}")

In [ ]:
import joblib
joblib.dump(clf, "model.joblib")
print("✅ Done. Download model.joblib")